In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp 

import qutip as qq

In [ ]:
# Helper Function

# Easier using 2->3


# Convertion helper 
# 1-> wave function  
# 2-> theta,phi 
# 3-> n,n location for hamiltonian
def C12(k,size):
    x=(k)%size[0]
    y=(k)//size[0]
    return x,y

def C13(k,j,size):
    i=k
    return i,j

def C21(pos,size):
    k=pos[0]+pos[1]*size[0]
    return k

def C23(pos,j,size):
    i=C21(pos,size)
    return i,j

def C31(i,j=0,n=0,m=0):
    return i

def C32(pos,size):
    x,y=C12(pos[0],size)
    return x,y

# Reconstruct Helper
def R12(par,size):
    i=np.array(range(size[0]*size[1]))
    val=np.zeros([size[1],size[0]],dtype=complex)
    x,y=C12(i,size)
    val[y,x]=par[i]
    return val

def R21(par,size):
    ipos=np.zeros([2,size[0]*size[1]],dtype=int)
    val=np.zeros(size[0]*size[1])
    for j in range(size[1]):
        for i in range(size[0]):
            ipos[0,j*size[0]+i]=i
            ipos[1,j*size[0]+i]=j
    k=C21(ipos,size)
    r=range(size[0]*size[1])
    val[k]=par[ipos[0,r],ipos[1,r]]
    return val


def FDiffTheta(H,pos,a,dt,j,size):
    b=a/(2*dt)

    if pos[1]!=0:
        H[j,C21(pos+[0,-1],size)]+=-b
    else:
        H[j,C21([pos[0],size[1]-1],size)]+=-b
    if pos[1]!=(size[1]-1):
        H[j,C21(pos+[0, 1],size)]+=b
    else:
        H[j,C21([pos[0],0],size)]+=b
    return H

def SDiffTheta(H,pos,a,dt,j,size):
    b=a/dt**2

    H[j,C21(pos,size)]-=2*b
    if pos[1]!=0:
        H[j,C21(pos+[0,-1],size)]+=b
    else:
        H[j,C21([pos[0],size[1]-1],size)]+=b
    if pos[1]!=(size[1]-1):
        H[j,C21(pos+[0, 1],size)]+=b
    else:
        H[j,C21([pos[0],size[1]-1],size)]+=b
    return H

def SDiffPhi(H,pos,a,dt,j,size):
    b=a/dt**2

    H[j,C21(pos,size)]-=2*b
    if pos[0]!=0:
        H[j,C21(pos+[-1,0],size)]+=b
    if pos[0]!=(size[0]-1):
        H[j,C21(pos+[ 1,0],size)]+=b
    return H

In [ ]:
# Main
N=80
M=40
NM=N*M
size=[N,M]
sphi=4*np.pi
stheta=np.pi
# N3=N*N*N

H=np.zeros([NM,NM],dtype=complex)
phi=np.linspace(-sphi,sphi,N)
theta=np.linspace(-stheta,stheta,M)
dp=phi[1]-phi[0]
dt=theta[1]-theta[0]

a=10000000000
Ect=a
Ecp=a
ng=0
phiE=0
El=a*5/1000
Ej=a

# for normal case
# 1<=x<n 1<=y<m
# H[j][k]+=
# H[j,C21(pos,size)]+=EEE
count=0
for j in range(M):
    for i in range(N):
        pos=np.array([i,j])
        H[count,C21(pos,size)]= +4*(Ect*ng)\
                                +El*(phi[i]-np.pi*phiE)**2\
                                +2*Ej*np.cos(phi[i])*np.cos(theta[j])
                                # +El*(phi[i]**2-2*phi[i]*np.pi*phiE+np.pi**2*phiE**2)\
        H=SDiffPhi(H,pos,-4*Ecp,dp,count,size)
        H=FDiffTheta(H,pos,-1j*8*Ect*ng,dt,count,size)
        H=SDiffTheta(H,pos,-4*Ect,dt,count,size)
        count+=1

# for edge case
# x<0,>=n  y<0,>=m dont add


# Solve using qutip
# QH = qq.Qobj(H)
# asort = QH.eigenenergies()
# asort,wf = QH.eigenstates()

# Solve using linalg normal
a,b=np.linalg.eig(H)
n=np.argsort(np.absolute(a))
# n=np.argsort(a)
asort=a[n]
wf=b[n]

# Solve using scipy eig fast
# Lupa namanya keknya bukan yg tridiag sama itu di ini gabisa
# asort,wf=sp.sparse.linalg.eigs(H,15)
# sparcematrix
# R12(H[3,:],size).real
# H[0,:]

In [94]:
# Functionized Main
def main(phiE,ng,size,sphi=4*np.pi,stheta=np.pi,Isincludestate=False):
    N,M=size
    NM=N*M

    H=np.zeros([NM,NM],dtype=complex)
    phi=np.linspace(-sphi,sphi,N)
    theta=np.linspace(-stheta,stheta,M)
    dp=phi[1]-phi[0]
    dt=theta[1]-theta[0]

    a=10000000000
    alpha=1
    beta=1
    Ect=a
    Ecp=alpha*a
    El=a*5/1000
    Ej=a

    count=0
    for j in range(M):
        for i in range(N):
            pos=np.array([i,j])
            H[count,C21(pos,size)]= +4*(Ect*ng)\
                                    +El*(phi[i]-np.pi*phiE)**2\
                                    +2*Ej*np.cos(phi[i])*np.cos(theta[j])
            H=SDiffPhi(H,pos,-4*Ecp,dp,count,size)
            H=FDiffTheta(H,pos,-1j*8*Ect*ng,dt,count,size)
            H=SDiffTheta(H,pos,-4*Ect,dt,count,size)
            count+=1

    # Solve using linalg normal
    if Isincludestate:
        a,b=np.linalg.eig(H)
        n=np.argsort(np.absolute(a))
        asort=a[n]
        wf=b[n]
        return asort,wf
    a=np.linalg.eigvals(H)
    n=np.argsort(np.absolute(a))
    asort=a[n]
    return asort
    
    # Solve using qutip
    # QH = qq.Qobj(H)
    # asort = QH.eigenenergies()
    # asort,wf = QH.eigenstates()


In [96]:
# Testing
# %matplotlib qt

n=4
size=[20,10]
Enum=10
En=np.zeros([n,Enum])
phiE=np.linspace(-0.5,0.5,n)
for i in range(n):
    En[i]=np.absolute(main(phiE[i],0.5,size)[:Enum])







# sN=int(N/4)
# sM=int(M/2)
# new=np.transpose(R12(np.abs(wf[:,1]),size).real)
# new=np.transpose(R12(np.absolute(b[0]),size).real)
# new=R12(np.abs(wf[:,1]),size).real
# plt.imshow(new[20:60,20:],cmap='hot', interpolation='nearest')
# plt.imshow(new,cmap='hot', interpolation='nearest')
# plt.show()
# asort[:20].real
# R12(H[NM-1,:],size)
# n=10
# m=20
# test1=np.zeros([n*m])
# test2=np.zeros([m,n,2])
# test3=np.zeros([n*m,n*m,2])
# for i in range(n*m):
#     test1[i]=i
# for i in range(n):
#     for j in range(m):
#         test2[j,i,0]=i
#         test2[j,i,1]=j
# for j in range(n*m):
#     for i in range(n*m):
#         test3[j,i,0]=i
#         test3[j,i,1]=j



In [5]:
# Testing Out
c1=np.zeros([n*m])
c12=np.zeros([n*m])
c13=np.zeros([n*m])
c2=np.zeros([m,n,2])
c21=np.zeros([m,n,2])
c23=np.zeros([m,n,2])
c3=np.zeros([n*m,n*m,2])
c31=np.zeros([n*m,n*m,2])
c32=np.zeros([n*m,n*m,2])

for i in range(n*m):
    a,b=C12(i,n,m)
    c21[b,a,0]=a
    c21[b,a,1]=b
for j in range(n*m):
    for i in range(n*m):
        a,b=C13(i,j,n,m)
        c31[b,a,0]=a
        c31[b,a,1]=b
for k in range(n*m):
    for j in range(m):
        for i in range(n):
            a,b=C23(i,j,k,n,m)
            c32[b,a,0]=a
            c32[b,a,1]=b
for j in range(n*m):
    for i in range(n*m):
        a,b=C32(i,j,n,m)
        c23[b,a,0]=a
        c23[b,a,1]=b

if np.array(c32 == test3).all():
    print("23")
if np.array(c31==test3).all():
    print("13")
if np.array(c21==test2).all():
    print("12")
# sparse eigs

NameError: name 'n' is not defined

In [ ]:
# Old code
# Wouldnt work for 2 variable system
def Diag(a,const,isAA,N=100,isComplex=False):    
    N3=N**3
    if isComplex:
        M=np.zeros([N3,N3],dtype=complex)
    else:
        M=np.zeros([N3,N3])

    if(isAA):
        for k in range(N):
            i,j=c23(k,const)
            M[i,j]=a
        return M
    for i in range(N):
        i,j=c23(k,const)
        M[i,j]=a
    return M

def FirstDiff(a,d,const,isAA,N=100,isComplex=False):
    N3=N**3
    if isComplex:
        M=np.zeros([N3,N3],dtype=complex)
    else:
        M=np.zeros([N3,N3])

    if(isAA):
       for i in range(N-1):    
           i,j=c23(k,const)
           M[i,j]=a/d/2      
           M[i,j]=-a/d/2
       return M
    for i in range(N-1):   
        i,j=c23(k,const)
        M[i,j]=a/d/2
        M[i,j]=-a/d/2
    return M
def SecondDiff(a,d,const,isAA,N=100,isComplex=False):
    N3=N**3
    if isComplex:
        M=np.zeros([N3,N3],dtype=complex)
    else:
        M=np.zeros([N3,N3])
    if(isAA):
        for k in range(N-1):
            i,j=c23(k,const)
            M[i,j]=-2*a/d**2
            M[i,j]=a/d**2
            M[i,j]=a/d**2
        I,J=c23(k,N-1)
        M[I,J]=a
        return M
    for k in range(N-1): 
        i,j=c23(const,k)
        M[i,j]=a/d**2
        M[i,j]=a/d**2
        M[i,j]=-2*a/d**2
    I,J=c23(k,N-1)
    M[I,J]=a
    return M

In [88]:
wf[0]

array([-0.00406947-3.79724870e-18j,  0.00079575+1.09279426e-17j,
       -0.00224617-1.96465057e-17j, ...,  0.02715093-9.58424556e-16j,
        0.01943449-2.47678356e-16j,  0.00070982-2.95709448e-16j],
      shape=(3200,))